In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')

In [14]:
#Load the CPI data
cpi_raw  = pd.read_excel('CPI_data_2008-2026.xlsx', header=0)

# now we see if it worked
print(f"Loaded {len(cpi_raw)} rows")

Loaded 789 rows


In [15]:
# Filtering for Total country rows only
# we use a boolean mask to reutrn a series of true and false values,
# one for each row in the datset.
mask_total = cpi_raw.iloc[:,6].astype(str).str.contains("Total country", na=False)
total_country = cpi_raw[mask_total].copy()

# Students basket - we just need the meal/living allowance components only
categories = {
    'Food': 'CPT01000',
    'Personal Care': 'CPT13100'
}

# we the pull each category row
basket_rows = {}
for name, code in categories.items():
    row = total_country[total_country.iloc[:, 2].astype(str).str.strip() == code.strip()] #row here is another mask
    #this time it filters column index 2 - the CPI column so that it gets the row where
    # the code matches so when the code = "CPT13100" it finds the FOOD row specifically
    basket_rows[name] = row
    print(f"{name} ({code}): {len(row)} row(s) found")
                                                        

Food (CPT01000): 1 row(s) found
Personal Care (CPT13100): 1 row(s) found


In [16]:
# Monthly columns start at index 11 (MO012008 onward)
monthly_cols = cpi_raw.columns[11:]

# Extract time series for each category
food_series = basket_rows['Food'].iloc[0, 11:].astype(float)
personal_care_series = basket_rows['Personal Care'].iloc[0, 11:].astype(float)

# Create a clean DataFrame with a proper date index
dates = pd.date_range(start='2008-01', periods=len(monthly_cols), freq='MS')

cpi_df = pd.DataFrame({
    'Food': food_series.values,
    'Personal Care': personal_care_series.values
}, index=dates)

# Filter to 2019 onwards - when our NSFAS story starts
cpi_df = cpi_df[cpi_df.index >= '2019-01-01']

print(cpi_df.shape)
print(cpi_df.head())
print(cpi_df.tail())

(90, 2)
            Food  Personal Care
2019-01-01  69.6           76.9
2019-02-01  69.6           76.8
2019-03-01  69.8           78.1
2019-04-01  70.0           77.6
2019-05-01  70.2           78.0
             Food  Personal Care
2026-02-01  104.2          101.2
2026-03-01  104.1          101.3
2026-04-01  104.8          101.2
2026-05-01  104.9          101.6
2026-06-01  105.3          101.6


In [7]:
#convert CPI index values to inflation multipliers relative to jan 2019
# this tells us how much more expensive things are compared to jan 2019
base_food = cpi_df['Food'].iloc[0] # Jan 2019 food index


['Total country' 'Rural areas' 'All urban areas' 'Western Cape'
 'Eastern Cape' 'Northern Cape' 'Free State' 'Kwazulu-Natal' 'North-West'
 'Gauteng' 'Mpumalanga' 'Limpopo' 'Rural Areas']


In [8]:
# Show all rows for Total country with their codes and descriptions
mask_total = cpi_raw.iloc[:, 6].astype(str) == 'Total country'
total_country = cpi_raw[mask_total].copy()

print(f"Total country rows: {len(total_country)}")
print()

# Print all codes and descriptions
for idx, row in total_country.iterrows():
    print(f"Code: {row.iloc[2]} | Description: {row.iloc[3]}")

Total country rows: 79

Code: CPI60001 | Description: Geographic indices
Code: CPT00000 | Description: All Items
Code: CPTD0001 | Description: Total country
Code: CPTD0002 | Description: Total country
Code: CPTD0003 | Description: Total country
Code: CPTD0004 | Description: Total country
Code: CPTD0005 | Description: Total country
Code: CPTD0006 | Description: Total country
Code: CPTD0007 | Description: Total country
Code: CPTD0008 | Description: Total country
Code: CPTD0009 | Description: Total country
Code: CPTD0010 | Description: Total country
Code: CPI60005 | Description: Pensioners
Code: CPT00021 | Description: Processed
Code: CPT00022 | Description: Unprocessed
Code: CPT01000 | Description: Food and non alcoholic beverages
Code: CPT01100 | Description: Food
Code: CPT01110 | Description: Cereal products
Code: CPT01120 | Description: Meat
Code: CPT01130 | Description: Fish and other seafood
Code: CPT01140 | Description: Milk, other dairy products and eggs
Code: CPT01150 | Descripti